In [6]:
import pandas as pd
import numpy as np

# build features using the same logic as notebook 03
movies = pd.read_csv("../data/ml-32m/movies.csv") # movieId,title,genres
links = pd.read_csv("../data/ml-32m/links.csv")

# use filtered_ratings2.csv for testing, which contains userId, tmdbId, rating, timestamp
# from user_id retrieve respective tmdb_id
ratings = pd.read_csv("../data/filtered_ratings2.csv")

# process ratings so that it is per user

movie_embeddings = pd.read_parquet("../data/movie_embeddings.parquet") # tmdbId, embedding_0, embedding_1, ..., embedding_n

# assume tmdb_id and respective rating is given to build user features for the userNN model
# per user
def build_user_features(ratings, movie_embeddings):
    """
    ratings (per user ratings) - DataFrame containing tmdbId, rating, timestamp
    movie_embeddings - DataFrame containing tmdbId and ALL movie embeddings
    """

    users_encoded = pd.DataFrame(index=[0])

    # genre avg ratings
    genre_columns = [
        "Action",
        "Adventure",
        "Animation",
        "Children",
        "Comedy",
        "Crime",
        "Documentary",
        "Drama",
        "Fantasy",
        "Film-Noir",
        "Horror",
        "IMAX",
        "Musical",
        "Mystery",
        "Romance",
        "Sci-Fi",
        "Thriller",
        "War",
        "Western",
    ]

    for genre in genre_columns:
        genre_movie_ids = movie_embeddings.index[movie_embeddings[f"movie_{genre}"] == 1]
        users_encoded[genre] = ratings[ratings.index.isin(genre_movie_ids)]["rating"].mean()

    # decade avg ratings
    decade_columns = [col for col in movie_embeddings.columns if col.startswith("movie_Decade_")]
    for decade in decade_columns:
        decade_movie_ids = movie_embeddings.index[movie_embeddings[decade] == 1]
        user_decade = decade.removeprefix("movie_")
        users_encoded[user_decade] = ratings[ratings.index.isin(decade_movie_ids)]["rating"].mean()

    users_encoded["avg_rating"] = ratings["rating"].mean()
    users_encoded["std_rating"] = ratings["rating"].std()
    users_encoded["count_rating"] = ratings["rating"].count()

    users_encoded = users_encoded.fillna(0)

    user_feature_frame = users_encoded.rename(
        columns={col: f"user_{col}" for col in users_encoded.columns}
    )

    return user_feature_frame

all_users = pd.DataFrame()
for user_id in ratings["userId"].unique():
    user_ratings = ratings[ratings["userId"] == user_id]
    user_feature_frame = build_user_features(user_ratings, movie_embeddings)
    all_users = pd.concat([all_users, user_feature_frame], ignore_index=True)

print("User feature frame:", all_users.shape)
print(all_users.head())
print("User feature columns:", all_users.columns.tolist())


User feature frame: (1000, 36)
   user_Action  user_Adventure  user_Animation  user_Children  user_Comedy  \
0     4.178571        4.062500        4.000000       4.000000     4.052632   
1     4.112903        4.145833        3.625000       3.000000     3.919355   
2     4.583333        4.500000        0.000000       4.833333     4.500000   
3     3.564103        3.774194        3.857143       4.200000     3.673913   
4     3.906977        3.775510        3.125000       3.090909     3.653061   

   user_Crime  user_Documentary  user_Drama  user_Fantasy  user_Film-Noir  \
0    4.352941             4.000    4.239130      3.833333        0.000000   
1    3.961538             3.900    3.700000      3.714286        4.500000   
2    4.750000             4.125    4.517857      4.285714        4.500000   
3    3.550000             3.000    3.647541      3.812500        4.333333   
4    4.000000             3.800    3.817204      3.689655        3.500000   

   ...  user_Decade_1960.0  user_Deca